# 01. 데이터 탐색

- 감정 데이터(5차년도_2차) csv와 방언 데이터(강원·경상 / 충청·전라·제주) 라벨링 json을 살펴봅니다.
- CPU 런타임으로 충분합니다. 그래프는 `reports/figures/`에 저장됩니다.
- ⚠️ 셀 출력에는 AI Hub 원문(전사 텍스트)이 포함됩니다. **커밋 전 '모든 출력 지우기'** 하세요.

In [ ]:
# [설정] 경로·컬럼명 — 여기만 수정하면 됩니다
REPO_URL = ""  # 비어 있으면 REPO_DIR에 이미 clone되어 있다고 가정
REPO_DIR = "/content/drive/MyDrive/mmdl/multimodal-dialect-emotion"
DATA_DIR = "/content/data"  # Colab 로컬 (00 또는 02의 압축 해제 셀을 먼저 실행)
FIG_DIR = f"{REPO_DIR}/reports/figures"

EMO_CSV_GLOB = f"{DATA_DIR}/emotion/emotion.csv"
EMO_LABEL_COL = "상황"    # 감정 라벨 컬럼 (아래 컬럼 목록 보고 수정)
EMO_TEXT_COL = "발화문"   # 텍스트(음성인식 결과) 컬럼

# 세트 이름 → 라벨링 json glob (2인발화 라벨링 폴더로 좁혀서 지정)
DIALECT_JSON_GLOBS = {
    "강원·경상": f"{DATA_DIR}/gs/label/**/*.json",
    "충청·전라·제주": f"{DATA_DIR}/cc/label/**/*.json",
}
MAX_JSON_PER_SET = 3000  # 세트당 읽을 json 수 (Drive 읽기가 느림). None이면 전부
AGE_REF_YEAR = 2021      # 연령 = 기준연도 - 출생연도 (녹음 연도에 맞게 수정)
SEED = 42

In [ ]:
# Drive 마운트, 저장소 경로 등록(src 임포트용), 그림 폴더 생성
from google.colab import drive
drive.mount("/content/drive")

import os, sys, subprocess
if not os.path.exists(REPO_DIR):
    assert REPO_URL, f"{REPO_DIR} 가 없습니다. REPO_URL을 채우거나 직접 clone하세요."
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR)
os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
# 한글 폰트 설치 및 matplotlib 설정, 그림 저장 함수
!apt-get -qq install -y fonts-nanum > /dev/null
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

def save(fig, name):
    fig.savefig(f"{FIG_DIR}/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("saved:", f"{FIG_DIR}/{name}.png")

In [ ]:
# 공용 모듈과 라이브러리 임포트
import glob, random, re
import pandas as pd
from src.data import read_csv_any, load_json, dialect_sentences, speakers
pd.set_option("display.max_colwidth", 80)

## 1. 감정 데이터 (5차년도_2차) csv

In [ ]:
# csv 파일 찾아서 읽기 (utf-8 / cp949 자동)
emo_csvs = sorted(glob.glob(EMO_CSV_GLOB, recursive=True))
print(emo_csvs)
emo = pd.concat([read_csv_any(p).assign(_file=os.path.basename(p)) for p in emo_csvs], ignore_index=True)
print(emo.shape)

In [ ]:
# 컬럼 목록과 타입·결측
emo.info()

In [ ]:
# 감정 라벨 분포 (EMO_LABEL_COL)
counts = emo[EMO_LABEL_COL].value_counts()
print(counts.to_frame().assign(비율=(counts / counts.sum()).round(3)))
fig, ax = plt.subplots(figsize=(7, 3.5))
counts.plot.bar(ax=ax)
ax.set_title(f"감정 데이터 라벨 분포 ({EMO_LABEL_COL})")
ax.set_ylabel("발화 수")
save(fig, "emotion_label_dist")

In [ ]:
# 평가자별 감정 컬럼(예: '1번 감정')이 있으면 다수결 라벨과 EMO_LABEL_COL 교차표
ann_cols = [c for c in emo.columns if re.fullmatch(r"\d번\s*감정", str(c))]
print("평가자 감정 컬럼:", ann_cols)
if ann_cols:
    emo["다수결감정"] = emo[ann_cols].mode(axis=1)[0]  # 동률이면 사전순 첫 번째
    display(pd.crosstab(emo[EMO_LABEL_COL], emo["다수결감정"]))

In [ ]:
# 텍스트(음성인식 결과) 컬럼 확인: 결측 비율, 글자 수 통계
text = emo[EMO_TEXT_COL]
print("결측 비율:", round(text.isna().mean(), 4))
print(text.dropna().str.len().describe())

In [ ]:
# 샘플 5개 출력
emo.sample(5, random_state=SEED)

## 2. 방언 데이터 라벨링 json

In [ ]:
# 세트별 json 파일 목록 (MAX_JSON_PER_SET만큼 무작위 샘플)
json_files = {}
for name, pattern in DIALECT_JSON_GLOBS.items():
    files = sorted(glob.glob(pattern, recursive=True))
    if MAX_JSON_PER_SET and len(files) > MAX_JSON_PER_SET:
        files = random.Random(SEED).sample(files, MAX_JSON_PER_SET)
    json_files[name] = files
    print(f"{name}: {len(files)}개 사용")

In [ ]:
# json 파일 하나의 구조 출력 (리스트는 첫 원소만)
def show_structure(x, key="root", depth=0):
    pad = "  " * depth
    if isinstance(x, dict):
        print(f"{pad}{key}: dict({len(x)})")
        for k, v in x.items():
            show_structure(v, k, depth + 1)
    elif isinstance(x, list):
        print(f"{pad}{key}: list[{len(x)}]")
        if x:
            show_structure(x[0], "[0]", depth + 1)
    else:
        print(f"{pad}{key}: {type(x).__name__} = {str(x)[:60]!r}")

sample_path = next(f for fs in json_files.values() for f in fs)
print(sample_path)
show_structure(load_json(sample_path))

In [ ]:
# 모든 json → 문장 단위 DataFrame(sent)과 화자 DataFrame(spk)
sent_rows, spk_rows = [], []
for name, files in json_files.items():
    for f in files:
        doc = load_json(f)
        base = os.path.basename(f)
        sent_rows += [dict(r, set=name, file=base) for r in dialect_sentences(doc)]
        spk_rows += [dict(s, set=name, file=base) for s in speakers(doc)]
sent = pd.DataFrame(sent_rows)
spk = pd.DataFrame(spk_rows)
print(sent.shape, spk.shape)
sent.head()

In [ ]:
# 지역(거주 시도)별 문장 수
by_region = sent.groupby(["set", "province"], dropna=False).size().rename("문장 수")
print(by_region)
fig, ax = plt.subplots(figsize=(8, 3.5))
by_region.droplevel(0).sort_values().plot.barh(ax=ax)
ax.set_title("지역별 문장 수")
save(fig, "dialect_sentences_by_region")

In [ ]:
# 감정 태그(tagType) 분포와 '무관' 비율 (지역별)
sent["emotion_tag"] = sent["emotion"].fillna("(태그 없음)")
tag_ratio = pd.crosstab(sent["province"], sent["emotion_tag"], normalize="index").round(3)
display(pd.crosstab(sent["province"], sent["emotion_tag"], margins=True))
display(tag_ratio)
print("전체 '무관' 비율:", round((sent["emotion"] == "무관").mean(), 3))
fig, ax = plt.subplots(figsize=(8, 3.5))
tag_ratio.plot.barh(stacked=True, ax=ax)
ax.set_title("지역별 감정 태그 비율")
ax.legend(bbox_to_anchor=(1, 1))
save(fig, "dialect_emotion_tag_ratio")

In [ ]:
# 문장별 억양 주석 개수 분포
print(sent.groupby("set")["n_intonations"].describe())
fig, ax = plt.subplots(figsize=(7, 3.5))
for name, g in sent.groupby("set"):
    g["n_intonations"].value_counts(normalize=True).sort_index().plot(ax=ax, marker="o", label=name)
ax.set_xlabel("문장당 억양 주석 수")
ax.set_ylabel("문장 비율")
ax.set_title("문장별 억양 주석 개수 분포")
ax.legend()
save(fig, "dialect_intonations_per_sentence")

In [ ]:
# 화자 연령 분포 (화자 id 기준 중복 제거)
# id가 파일 안에서만 쓰는 번호(1, 2 등)라면 drop_duplicates(["file", "id"])로 바꾸세요
uniq = spk.drop_duplicates("id") if "id" in spk else spk
uniq = uniq.assign(age=AGE_REF_YEAR - pd.to_numeric(uniq["birthYear"], errors="coerce"))
print(uniq.groupby("residenceProvince")["age"].describe())
fig, ax = plt.subplots(figsize=(7, 3.5))
for prov, g in uniq.groupby("residenceProvince"):
    g["age"].plot.hist(ax=ax, bins=range(40, 101, 5), alpha=0.5, label=prov)
ax.set_xlabel(f"연령 ({AGE_REF_YEAR}년 기준)")
ax.set_title("화자 연령 분포")
ax.legend()
save(fig, "dialect_speaker_age")

In [ ]:
# STT 결과 vs 사람 전사(방언/표준어) 비교 샘플 5개
cols = ["set", "province", "emotion", "dialect", "standard", "stt"]
sent[sent["stt"] != ""].sample(5, random_state=SEED)[cols]